In [ ]:


from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL = "Qwen/Qwen2.5-32B-Instruct"
tk = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

# Вариант 1 — несколько GPU (если на кластере 2+ A100 80GB)
md = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # bfloat16 стабильнее чем float16 для больших моделей
    trust_remote_code=True
).eval()


# ═══════════════════════════════════════════════════════════
# Ячейка 3 — Данные
# ═══════════════════════════════════════════════════════════
import pandas as pd
import os, re
from io import StringIO
from tqdm import tqdm

tagged_df = pd.read_excel(f'batchs/S_p_m.xlsx')
found_df  = pd.read_excel(f'batchs/found_evidence.xlsx')
found_ids = set(found_df['id'])
model_df  = tagged_df[~tagged_df['id'].isin(found_ids)].reset_index(drop=True)
print(f"Для модели: {len(model_df)} строк")

# ═══════════════════════════════════════════════════════════
# Ячейка 4 — Промпт и генерация
# ═══════════════════════════════════════════════════════════
SYSTEM = """You are a data augmentation expert for NLP research.

TASK: Given a QUESTION, ANSWER, and TABLE — return a modified table (T-) where the correct answer CANNOT be derived.

STRICT RULES:
1. Output ONLY raw CSV. No markdown, no explanations, no ```csv blocks.
2. Keep ALL column headers identical.
3. Keep the EXACT same number of rows.
4. Change only cell VALUES, not structure.
5. The answer must be impossible to derive from T-.
6. T- must look realistic.

STRATEGIES:
- Replace the answer value with a different plausible value
- Swap values between rows to break ordering
- Change numeric values so computations yield wrong results
- Replace category values so filters return wrong results

EXAMPLES:

QUESTION: what was the last year where this team was part of the usl a-league?
ANSWER: 2004
TABLE:
Year,Division,League
2001,2,USL A-League
2002,2,USL A-League
2003,2,USL A-League
2004,2,USL A-League
2005,2,USL First Division
OUTPUT:
Year,Division,League
2001,2,USL A-League
2002,2,USL A-League
2003,2,USL A-League
2004,2,USL First Division
2005,2,USL First Division

QUESTION: which country had the highest gdp in billions?
ANSWER: Nigeria
TABLE:
Country,GDP (billions)
Nigeria,375.8
Kenya,98.3
Ghana,65.5
OUTPUT:
Country,GDP (billions)
Nigeria,200.1
Kenya,412.1
Ghana,65.5

QUESTION: how many more passengers flew to los angeles than to saskatoon?
ANSWER: 12467
TABLE:
City,Passengers
Los Angeles,14749
Saskatoon,2282
OUTPUT:
City,Passengers
Los Angeles,10000
Saskatoon,2282

QUESTION: which team won previous to crettyard?
ANSWER: Wolfe Tones
TABLE:
Team,Year
Crettyard,2005
Wolfe Tones,2004
Dundalk Gaels,2003
OUTPUT:
Team,Year
Crettyard,2005
Wolfe Tones,2003
Dundalk Gaels,2004"""


def extract_csv(response: str, expected_cols: list) -> str | None:
    """Извлекает CSV из ответа модели — берёт последний валидный блок."""
    # убираем markdown
    response = re.sub(r'```[a-z]*\n?', '', response).replace('```', '').strip()

    # ищем строку с заголовками
    header = ','.join(expected_cols)
    if header not in response:
        # пробуем найти любую строку похожую на заголовок
        lines = response.split('\n')
        for i, line in enumerate(lines):
            if all(col.lower() in line.lower() for col in expected_cols[:2]):
                response = '\n'.join(lines[i:])
                break

    return response


def generate_t_minus(question: str, answer: str, tbl: pd.DataFrame) -> pd.DataFrame | None:
    # обрезаем большие таблицы
    tbl_input = tbl.head(20) if len(tbl) > 20 else tbl

    user_msg = (
        f"QUESTION: {question}\n"
        f"ANSWER: {answer}\n"
        f"TABLE:\n{tbl_input.to_csv(index=False)}\n"
        f"OUTPUT:"
    )
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": user_msg}
    ]

    text   = tk.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tk([text], return_tensors="pt").to(md.device)

    with torch.no_grad():
        out = md.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tk.eos_token_id
        )

    response = tk.decode(
        out[0][len(inputs.input_ids[0]):],
        skip_special_tokens=True
    ).strip()

    # парсинг
    try:
        csv_text = extract_csv(response, list(tbl_input.columns))
        t_minus  = pd.read_csv(StringIO(csv_text))

        # проверки
        if list(t_minus.columns) != list(tbl_input.columns):
            return None, f"columns mismatch: {list(t_minus.columns)}"
        if len(t_minus) != len(tbl_input):
            return None, f"rows mismatch: {len(t_minus)} vs {len(tbl_input)}"
        if tbl_input.to_csv(index=False) == t_minus.to_csv(index=False):
            return None, "identical to original"

        return t_minus, None
    except Exception as e:
        return None, str(e)


# ═══════════════════════════════════════════════════════════
# Ячейка 5 — Тест на 10 примерах
# ═══════════════════════════════════════════════════════════
for _, row in model_df.head(10).iterrows():
    try:
        tbl = pd.read_csv(f"{BASE}/data/{row['context']}",
                          quotechar='"', on_bad_lines='skip', engine='python')
        if tbl.empty:
            print(f"❌ id={row['id']}: ПУСТАЯ ТАБЛИЦА")
            continue

        t_minus, err = generate_t_minus(row['utterance'], row['targetValue'], tbl)

        if t_minus is None:
            print(f"❌ id={row['id']}: {err}")
        else:
            print(f"\n{'='*60}")
            print(f"✅ id={row['id']}")
            print(f"S:      {row['utterance']}")
            print(f"Answer: {row['targetValue']}")
            print(f"\n--- ОРИГИНАЛ ---\n{tbl.head(5).to_string(index=False)}")
            print(f"\n--- T- ---\n{t_minus.head(5).to_string(index=False)}")

    except Exception as e:
        print(f"❌ id={row['id']}: {type(e).__name__} - {e}")


# ═══════════════════════════════════════════════════════════
# Ячейка 6 — Полный прогон с чекпоинтом
# ═══════════════════════════════════════════════════════════
os.makedirs(f'{BASE}/data/tables', exist_ok=True)
CHECKPOINT_FILE = f'{BASE}/batchs/t_minus_results.xlsx'
FAILED_FILE     = f'{BASE}/batchs/t_minus_failed.xlsx'
SAVE_EVERY      = 50

# загружаем чекпоинт если есть
results, failed = [], []
done_ids = set()
if os.path.exists(CHECKPOINT_FILE):
    done_df  = pd.read_excel(CHECKPOINT_FILE)
    results  = done_df.to_dict('records')
    done_ids = set(done_df['id'])
    print(f"Чекпоинт: уже готово {len(done_ids)} строк")

for _, row in tqdm(model_df.iterrows(), total=len(model_df)):
    if row['id'] in done_ids:
        continue
    try:
        tbl = pd.read_csv(f"{BASE}/data/{row['context']}",
                          quotechar='"', on_bad_lines='skip', engine='python')
        if tbl.empty:
            failed.append({**row.to_dict(), 'error': 'empty table'})
            continue

        t_minus, err = generate_t_minus(row['utterance'], row['targetValue'], tbl)

        if t_minus is None:
            failed.append({**row.to_dict(), 'error': err})
        else:
            path = f"{BASE}/data/tables/{row['id']}_negative.csv"
            t_minus.to_csv(path, index=False, encoding='utf-8')
            results.append({**row.to_dict(), 'T_minus_path': path})

    except Exception as e:
        failed.append({**row.to_dict(), 'error': str(e)})

    # чекпоинт каждые 50 строк
    if (len(results) + len(failed)) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_excel(CHECKPOINT_FILE, index=False)
        pd.DataFrame(failed).to_excel(FAILED_FILE,      index=False)

# финальное сохранение
pd.DataFrame(results).to_excel(CHECKPOINT_FILE, index=False)
pd.DataFrame(failed).to_excel(FAILED_FILE,      index=False)
print(f"✅ Успешно: {len(results)}")
print(f"⚠️  Failed: {len(failed)}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL = "Qwen/Qwen2.5-32B-Instruct"
tk = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

# Вариант 1 — несколько GPU (если на кластере 2+ A100 80GB)
md = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # bfloat16 стабильнее чем float16 для больших моделей
    trust_remote_code=True
).eval()

# Вариант 2 — одна GPU с квантизацией 4bit (если одна A100 40/80GB)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
md = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
).eval()